# Thick Residual Hybrid Stacking: Autoencoder + LightGBM for CVD Prediction

## Architecture Overview

A sophisticated neural-symbolic hybrid approach:

1. **Healthy Baseline Learning**: Train a PyTorch Autoencoder on healthy microbiome only
2. **Feature Extraction**: Extract 64D latent vectors + reconstruction errors from entire dataset
3. **Massive Concatenation**: Combine original OTUs + latent features + reconstruction errors
4. **Tree-Based Meta-Classifier**: LightGBM learns from rich feature representation
5. **Interpretability**: SHAP analysis reveals which learned features matter for CVD prediction

## Workflow
- Input: CLR-transformed OTU matrix, metadata with health status
- Output: Cross-validated CVD predictions with feature importance rankings
- Target: Identify if neural latent representations improve over raw features alone

In [ ]:
# ============================================================================
# IMPORTS: PyTorch, ML, and Visualization Libraries
# ============================================================================
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# PyTorch for neural network components
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW

# Machine Learning and Evaluation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
import lightgbm as lgb

# SHAP for interpretability
import shap

# Visualization
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ PyTorch device: {device}")
print(f"✓ All imports successful")

## Section 1: PyTorch Autoencoder Architecture

Define a symmetric Autoencoder with:
- **Encoder**: Input → 256 (BatchNorm, ReLU, Dropout 0.3) → 64 (Latent)
- **Decoder**: 64 → 256 (BatchNorm, ReLU, Dropout 0.3) → Output
- Goal: Learn to reconstruct healthy microbiome baseline

In [ ]:
class HealthyMicrobiomeAE(nn.Module):
    """
    Symmetric Autoencoder for learning healthy microbiome baseline.
    
    Architecture:
    - Encoder: Input_Dim → 256 → 64 (latent)
    - Decoder: 64 → 256 → Input_Dim
    
    Each dense layer is followed by BatchNorm, ReLU, and Dropout (0.3)
    """
    
    def __init__(self, input_dim, latent_dim=64, dropout_rate=0.3):
        super(HealthyMicrobiomeAE, self).__init__()
        
        # ===== ENCODER =====
        self.encoder = nn.Sequential(
            # Input -> 256
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 256 -> 64 (Latent space)
            nn.Linear(256, latent_dim),
            nn.BatchNorm1d(latent_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        # ===== DECODER =====
        self.decoder = nn.Sequential(
            # 64 -> 256
            nn.Linear(latent_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 256 -> Input_Dim (reconstruction)
            nn.Linear(256, input_dim)
        )
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
    
    def encode(self, x):
        """Extract latent representation."""
        return self.encoder(x)
    
    def decode(self, z):
        """Reconstruct from latent representation."""
        return self.decoder(z)
    
    def forward(self, x):
        """Full autoencoder forward pass: encode then decode."""
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

# Test instantiation
print("=" * 80)
print("AUTOENCODER ARCHITECTURE")
print("=" * 80)
print("\nHealthyMicrobiomeAE architecture:")
print("  Input Dimension:   variable (OTU count)")
print("  Hidden Dimension:  256")
print("  Latent Dimension:  64")
print("  Dropout:           0.3")
print("\nSummary:")
print("  - Encoder: Input → 256 (BN, ReLU, DO0.3) → 64 (Latent)")
print("  - Decoder: 64 → 256 (BN, ReLU, DO0.3) → Input")
print("  - Loss: MSELoss between input and reconstruction")
print("  - Optimizer: AdamW")
print("✓ Autoencoder class defined successfully")

## Section 2: Data Loading & Setup

Assume `otu_clr` (CLR-transformed OTU matrix) and `meta` (metadata with age, health, CVD status) are already loaded.

In [ ]:
print("=" * 80)
print("DATA LOADING & VALIDATION")
print("=" * 80)

# Assume otu_clr and meta are already in memory
# otu_clr: DataFrame (n_samples, n_otus) - CLR-transformed
# meta: DataFrame with columns ['age', 'health', 'cardiovascular_disease']

# Verify data exists and shapes match
assert otu_clr.shape[0] == meta.shape[0], "Sample count mismatch!"
n_samples, n_otus = otu_clr.shape

print(f"\nDataset loaded:")
print(f"  Samples: {n_samples}")
print(f"  OTU features: {n_otus}")
print(f"  Metadata columns: {meta.columns.tolist()}")

# Extract health status
is_healthy = meta['health'].values == 'y'
n_healthy = is_healthy.sum()
n_sick = (~is_healthy).sum()

print(f"\nHealth status:")
print(f"  Healthy: {n_healthy}")
print(f"  Non-healthy: {n_sick}")

# Convert OTU data to float32 numpy for PyTorch
X_otu_np = otu_clr.values.astype(np.float32)
print(f"\n✓ Data ready for training")

## Section 3: Train Autoencoder on Healthy Cohort

The Autoencoder learns to reconstruct a "normal" healthy microbiome. Training uses:
- **Data**: Only healthy individuals (meta['health'] == 'y')
- **Loss**: MSELoss between input and reconstruction
- **Optimizer**: AdamW with lr=1e-3
- **Epochs**: 30
- **Batch Size**: 32 (or adaptive to data size)

In [ ]:
print("=" * 80)
print("AUTOENCODER TRAINING (HEALTHY COHORT ONLY)")
print("=" * 80)

# Extract healthy cohort only
X_healthy_np = X_otu_np[is_healthy]

print(f"\nTraining set: {len(X_healthy_np)} healthy samples")

# Create PyTorch dataset and dataloader
batch_size = min(32, max(8, len(X_healthy_np) // 10))  # Adaptive batch size
X_healthy_tensor = torch.tensor(X_healthy_np, dtype=torch.float32)
dataset_healthy = TensorDataset(X_healthy_tensor)
dataloader_healthy = DataLoader(dataset_healthy, batch_size=batch_size, shuffle=True)

print(f"Batch size: {batch_size}")
print(f"Number of batches per epoch: {len(dataloader_healthy)}")

# Initialize autoencoder
ae = HealthyMicrobiomeAE(input_dim=n_otus, latent_dim=64, dropout_rate=0.3).to(device)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = AdamW(ae.parameters(), lr=1e-3, weight_decay=1e-5)

# Training loop
n_epochs = 30
ae.train()
losses = []

print(f"\nTraining for {n_epochs} epochs...")

for epoch in range(n_epochs):
    epoch_loss = 0.0
    for batch_data, in dataloader_healthy:
        batch_data = batch_data.to(device)
        
        # Forward pass
        x_recon, z = ae(batch_data)
        loss = criterion(x_recon, batch_data)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * batch_data.shape[0]
    
    epoch_loss /= len(X_healthy_np)
    losses.append(epoch_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:2d}/{n_epochs}: Loss = {epoch_loss:.6f}")

print(f"\n✓ Autoencoder training complete")
print(f"  Final loss: {losses[-1]:.6f}")
print(f"  Loss history: min={min(losses):.6f}, max={max(losses):.6f}")

## Section 4: Feature Extraction ("Thick Residuals")

Extract latent features and reconstruction errors for the *entire* dataset:
- **Latent Features**: 64-dimensional embeddings from encoder (captures learned healthy baseline)
- **Reconstruction Error**: MSE between input and output for each sample (measures deviation from healthy norm)

In [ ]:
print("=" * 80)
print("FEATURE EXTRACTION: LATENT REPRESENTATIONS & RECONSTRUCTION ERRORS")
print("=" * 80)

# Set model to evaluation mode
ae.eval()

# Extract features for the ENTIRE dataset (healthy + CVD)
X_full_tensor = torch.tensor(X_otu_np, dtype=torch.float32).to(device)

print(f"\nExtracting features for all {n_samples} samples...")

with torch.no_grad():
    # Full pass through autoencoder
    X_recon, latent_features = ae(X_full_tensor)
    
    # Calculate reconstruction error (MSE per sample)
    reconstruction_error = torch.mean((X_recon - X_full_tensor) ** 2, dim=1)  # Shape: (n_samples,)

# Move to CPU and convert to numpy
latent_features_np = latent_features.cpu().numpy()  # Shape: (n_samples, 64)
reconstruction_error_np = reconstruction_error.cpu().numpy()  # Shape: (n_samples,)

print(f"\n✓ Features extracted:")
print(f"  Latent features shape: {latent_features_np.shape} (64-dimensional)")
print(f"  Reconstruction error shape: {reconstruction_error_np.shape}")

print(f"\nLatent features statistics:")
print(f"  Mean: {latent_features_np.mean():.6f}")
print(f"  Std: {latent_features_np.std():.6f}")
print(f"  Min: {latent_features_np.min():.6f}, Max: {latent_features_np.max():.6f}")

print(f"\nReconstruction error statistics:")
print(f"  Mean: {reconstruction_error_np.mean():.6f}")
print(f"  Std: {reconstruction_error_np.std():.6f}")
print(f"  Min: {reconstruction_error_np.min():.6f}, Max: {reconstruction_error_np.max():.6f}")

# Compare reconstruction errors between healthy and CVD
ae_error_healthy = reconstruction_error_np[is_healthy]
ae_error_cvd = reconstruction_error_np[~is_healthy]

print(f"\nReconstruction error by group:")
print(f"  Healthy mean: {ae_error_healthy.mean():.6f} ± {ae_error_healthy.std():.6f}")
print(f"  CVD mean: {ae_error_cvd.mean():.6f} ± {ae_error_cvd.std():.6f}")
print(f"  Difference (CVD - Healthy): {ae_error_cvd.mean() - ae_error_healthy.mean():.6f}")
print(f"  → CVD patients have {'higher' if ae_error_cvd.mean() > ae_error_healthy.mean() else 'lower'} reconstruction errors")

## Section 5: Massive Feature Concatenation

Create the "thick" hybrid feature matrix by stacking:
1. Original CLR-transformed OTUs (~942 features)
2. 64-dimensional latent vectors
3. 1-dimensional reconstruction error

**Total dimensions**: ~1,007 features per sample

In [ ]:
print("=" * 80)
print("MASSIVE FEATURE CONCATENATION: BUILDING HYBRID MATRIX")
print("=" * 80)

# Create feature matrix by concatenating:
# 1. Original OTUs (copy)
# 2. 64 latent features
# 3. 1 reconstruction error

X_hybrid = otu_clr.copy()

# Add latent features with descriptive names
for i in range(latent_features_np.shape[1]):
    X_hybrid[f'latent_{i:02d}'] = latent_features_np[:, i]

# Add reconstruction error
X_hybrid['reconstruction_error'] = reconstruction_error_np

print(f"\n✓ Hybrid feature matrix constructed:")
print(f"  Shape: {X_hybrid.shape}")
print(f"  Breakdown:")
print(f"    - Original OTU features: {n_otus}")
print(f"    - Latent features: 64")
print(f"    - Reconstruction error: 1")
print(f"    - Total: {X_hybrid.shape[1]}")

print(f"\nFeature matrix columns (last 10):")
print(X_hybrid.columns.tolist()[-10:])

print(f"\nData types: {X_hybrid.dtypes.unique()}")
print(f"Missing values: {X_hybrid.isnull().sum().sum()}")

print(f"\nExample hybrid features (first 5 rows, last 5 columns):")
print(X_hybrid.iloc[:5, -5:])

## Section 6: Target Variable & LightGBM Classification

Create binary target for CVD prediction and train LightGBM with 5-fold StratifiedKFold cross-validation.

In [ ]:
print("=" * 80)
print("TARGET VARIABLE & LIGHTGBM CLASSIFICATION")
print("=" * 80)

# Create target variable
# CVD = 1 if cardiovascular_disease == 'I have this condition', else 0
if 'cardiovascular_disease' in meta.columns:
    y = (meta['cardiovascular_disease'] == 'I have this condition').astype(int)
    print("\n✓ Target from 'cardiovascular_disease' column")
else:
    # Fallback: use health column
    y = (meta['health'] != 'y').astype(int)
    print("\n✓ Target from 'health' column (health != 'y')")

print(f"Target distribution:")
print(f"  No CVD (y=0): {(y == 0).sum()}")
print(f"  CVD (y=1): {(y == 1).sum()}")
print(f"  Class imbalance: {(y == 0).sum() / (y == 1).sum():.1f}:1")

# 5-Fold Stratified Cross-Validation
print(f"\n" + "=" * 80)
print("5-FOLD STRATIFIED CROSS-VALIDATION WITH LIGHTGBM")
print("=" * 80)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []
all_y_val = []
all_y_pred_proba = []

print(f"\nTraining LGBMClassifier with:")
print(f"  - class_weight='balanced' (handles imbalance)")
print(f"  - n_estimators=200")
print(f"  - n_jobs=-1 (parallel)")
print(f"\n")

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_hybrid, y), start=1):
    # Split data
    X_train, X_val = X_hybrid.iloc[train_idx], X_hybrid.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Initialize LightGBM with class_weight='balanced'
    clf = lgb.LGBMClassifier(
        class_weight='balanced',
        n_estimators=200,
        num_leaves=31,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    # Train and predict
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    y_pred_proba = clf.predict_proba(X_val)[:, 1]
    
    # Calculate metrics
    ba = balanced_accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_pred_proba)
    
    cv_results.append({'fold': fold_idx, 'balanced_accuracy': ba, 'auc_roc': auc})
    all_y_val.extend(y_val.values)
    all_y_pred_proba.extend(y_pred_proba)
    
    # Store final model for SHAP
    if fold_idx == 5:
        final_clf = clf
        X_train_final = X_train
        X_val_final = X_val
        X_hybrid_for_shap = X_hybrid
        y_for_shap = y
    
    print(f"Fold {fold_idx}: BA = {ba:.4f}, AUC-ROC = {auc:.4f}")

# Aggregate results
print(f"\n" + "-" * 80)
cv_df = pd.DataFrame(cv_results)
mean_ba = cv_df['balanced_accuracy'].mean()
std_ba = cv_df['balanced_accuracy'].std()
mean_auc = cv_df['auc_roc'].mean()
std_auc = cv_df['auc_roc'].std()

print(f"\n✓ CROSS-VALIDATION RESULTS:")
print(f"  Balanced Accuracy: {mean_ba:.4f} ± {std_ba:.4f}")
print(f"  AUC-ROC:           {mean_auc:.4f} ± {std_auc:.4f}")

# Check against targets
if mean_ba > 0.70 and mean_auc > 0.75:
    print(f"\n✓✓✓ TARGET METRICS MET (BA > 70%, AUC > 75%)")
else:
    if mean_ba > 0.70:
        print(f"\n✓ BA target met")
    if mean_auc > 0.75:
        print(f"\n✓ AUC target met")

## Section 7: SHAP Interpretability Analysis

After cross-validation, fit LightGBM on the full dataset and use SHAP to:
1. Identify which features are most important
2. Specifically check if learned features (latent_* or reconstruction_error) rank in top 20
3. Visualize feature impact on CVD predictions

In [ ]:
print("=" * 80)
print("SHAP INTERPRETABILITY ANALYSIS")
print("=" * 80)

# Re-fit LightGBM on ENTIRE dataset for interpretability
print("\nFitting LightGBM on full dataset for SHAP analysis...")
final_model = lgb.LGBMClassifier(
    class_weight='balanced',
    n_estimators=200,
    num_leaves=31,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
final_model.fit(X_hybrid_for_shap, y_for_shap)
print("✓ Model fitted on full dataset")

# Initialize SHAP TreeExplainer
print("\nInitializing SHAP TreeExplainer...")
explainer = shap.TreeExplainer(final_model)

# Compute SHAP values
print("Computing SHAP values (this may take a moment)...")
shap_values = explainer.shap_values(X_hybrid_for_shap)

# For binary classification, shap_values is a list [class_0, class_1]
# Use class 1 (CVD positive class) for interpretability
if isinstance(shap_values, list):
    shap_values_cvd = shap_values[1]
    print(f"✓ SHAP values computed for CVD class (class 1)")
else:
    shap_values_cvd = shap_values
    print(f"✓ SHAP values computed")

print(f"  Shape: {shap_values_cvd.shape}")

# Calculate mean absolute SHAP values for global feature importance
mean_abs_shap = np.abs(shap_values_cvd).mean(axis=0)
feature_importance_df = pd.DataFrame({
    'feature': X_hybrid_for_shap.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print(f"\n✓ Feature importance ranked")
print(f"  Top 30 features:\n")

# Identify learned features in top 20/50
top_20_features = feature_importance_df.head(20)
learned_in_top_20 = top_20_features[
    top_20_features['feature'].str.startswith(('latent_', 'reconstruction_'))
]

print("Top 20 most important features:")
for idx, row in top_20_features.iterrows():
    feat_name = row['feature']
    shap_val = row['mean_abs_shap']
    
    # Mark learned features
    marker = ""
    if feat_name.startswith('latent_'):
        marker = " ← LATENT FEATURE"
    elif feat_name == 'reconstruction_error':
        marker = " ← RECONSTRUCTION ERROR"
    
    print(f"  {idx+1:2d}. {feat_name:25s} : {shap_val:.6f}{marker}")

# Summary of learned features in top ranks
print(f"\n" + "=" * 80)
print("LEARNED FEATURES ANALYSIS")
print("=" * 80)

latent_in_top50 = feature_importance_df.head(50)[
    feature_importance_df.head(50)['feature'].str.startswith('latent_')
].shape[0]

recon_error_rank = feature_importance_df[
    feature_importance_df['feature'] == 'reconstruction_error'
].index.tolist()

print(f"\nLatent features (64 total):")
print(f"  - In top 20: {learned_in_top_20[learned_in_top_20['feature'].str.startswith('latent_')].shape[0]}")
print(f"  - In top 50: {latent_in_top50}")

if len(recon_error_rank) > 0:
    rank = recon_error_rank[0] + 1
    importance = feature_importance_df.iloc[recon_error_rank[0]]['mean_abs_shap']
    print(f"\nReconstruction Error:")
    print(f"  - Rank: #{rank} out of {len(feature_importance_df)}")
    print(f"  - Mean |SHAP|: {importance:.6f}")
    if rank <= 20:
        print(f"  ✓ In TOP 20!")
    elif rank <= 50:
        print(f"  ✓ In TOP 50")
else:
    print(f"\nReconstruction Error: Not found (unexpected)")

# SHAP summary plot (bar)
print(f"\nGenerating SHAP summary plot (bar)...")
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_cvd, X_hybrid_for_shap, plot_type="bar", show=False)
plt.title('SHAP Feature Importance: Thick Residual Hybrid Stacking\n(Top 20 features for CVD prediction)', 
          fontsize=13, fontweight='bold')
plt.xlabel('Mean |SHAP value|', fontsize=12)
plt.tight_layout()
plt.savefig('thick_residual_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: thick_residual_shap_bar.png")

# SHAP beeswarm plot
print(f"\nGenerating SHAP beeswarm plot...")
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values_cvd, X_hybrid_for_shap, plot_type="violin", show=False)
plt.title('SHAP Feature Impact Distribution: Thick Residual Hybrid Stacking\n(Color: feature value, Position: impact on CVD prediction)', 
          fontsize=13, fontweight='bold')
plt.xlabel('SHAP value', fontsize=12)
plt.tight_layout()
plt.savefig('thick_residual_shap_violin.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: thick_residual_shap_violin.png")

## Section 8: Results Summary & Insights

Analysis complete. Review here:
- Cross-validation metrics and target achievement
- Ranking of learned (neural) features vs raw OTU features
- Whether the autoencoder's latent representations add predictive value

In [ ]:
print("\n" + "=" * 80)
print("THICK RESIDUAL HYBRID STACKING: COMPLETE")
print("=" * 80)

summary_text = f"""
EXECUTION SUMMARY
─────────────────────────────────────────────────────────────────────────────

Dataset Input:
  Samples: {n_samples}
  OTU features: {n_otus}
  Healthy: {n_healthy}, Non-healthy: {n_sick}

Autoencoder Training (Healthy Only):
  Training samples: {len(X_healthy_np)}
  Epochs: 30
  Batch size: {batch_size}
  Final training loss: {losses[-1]:.6f}

Feature Extraction ("Thick Residuals"):
  Latent features dimension: 64
  Reconstruction error: 1D
  
Hybrid Feature Matrix:
  Total features: {X_hybrid.shape[1]}
    - Original OTUs: {n_otus}
    - Latent features: 64
    - Reconstruction error: 1

Classification Performance (5-Fold StratifiedKFold):
  Balanced Accuracy: {mean_ba:.4f} ± {std_ba:.4f}
  AUC-ROC:           {mean_auc:.4f} ± {std_auc:.4f}

Learned Features Ranking (SHAP):
  Latent features in top 20: {learned_in_top_20[learned_in_top_20['feature'].str.startswith('latent_')].shape[0]}
  Latent features in top 50: {latent_in_top50}
  Reconstruction error rank: #{recon_error_rank[0] + 1 if len(recon_error_rank) > 0 else 'N/A'} out of {len(feature_importance_df)}

Key Insight:
  {'✓ LEARNED FEATURES ARE VALUABLE!' if (learned_in_top_20.shape[0] > 0 or (len(recon_error_rank) > 0 and recon_error_rank[0] < 50)) else '⚠ Learned features ranked lower than raw OTUs. Consider:'}
  - Autoencoder captures meaningful healthy baseline patterns
  - {f'{recon_error_rank[0] + 1}' if len(recon_error_rank) > 0 else 'Reconstruction error'} ranks {'highly' if len(recon_error_rank) > 0 and recon_error_rank[0] < 50 else 'moderately'}
  - {f'{learned_in_top_20[learned_in_top_20["feature"].str.startswith("latent_")].shape[0]} latent features' if learned_in_top_20.shape[0] > 0 else 'Few latent features'} are in top 20 important features

Outputs Generated:
  ✓ Cross-validation metrics
  ✓ SHAP summary plots:
    - thick_residual_shap_bar.png (feature importance ranking)
    - thick_residual_shap_violin.png (feature value distributions)

Next Steps:
  1. Compare performance against:
     - Raw OTU only + LightGBM (baseline)
     - OTU + reconstruction_error only (simpler version)
  2. Ablation study: Remove learned features and measure performance drop
  3. Fine-tune Autoencoder architecture (latent_dim, layers, dropout)
  4. Cross-cohort validation (AGP → GGMP generalization)
  5. Export latent features for downstream tasks (clustering, visualization)

─────────────────────────────────────────────────────────────────────────────
"""

print(summary_text)
print("✓ Thick Residual Hybrid Stacking Pipeline Complete!")

## Architecture Justification & Technical Notes

### Why "Thick Residual"?

The term "Thick Residual" refers to the approach of stacking **learned latent representations** alongside **raw input features**:

- **Traditional residual**: Output + Residual = Target
- **Thick residual here**: 
  - Autoencoder learns compressed "healthy baseline" in latent space
  - Model gets both raw signals (OTUs) and learned signals (latents + reconstruction error)
  - Tree-based explainer (SHAP) reveals which learned patterns matter most

### Key Design Choices

1. **Train on Healthy Only**: Autoencoder learns what "normal" looks like, making deviations (CVD) detectable
2. **64-dimensional Latents**: Balanced compression (942 OTUs → 64D) for computational efficiency
3. **MSE Loss**: Reconstruction error directly measures deviation from healthy baseline
4. **Massive Concatenation**: Tree-based models handle high-dimensional data well; let the model learn feature interactions
5. **SHAP for Interpretation**: Reveals whether learned features or raw OTUs drive predictions

### Hyperparameter Tuning Ideas

| Component | Current | Try |
|-----------|---------|-----|
| Latent dimension | 64 | 32, 128 |
| Hidden layer | 256 | 128, 512 |
| Dropout | 0.3 | 0.2, 0.5 |
| AE epochs | 30 | 50, 100 |
| LGB n_estimators | 200 | 100, 500 |
| LGB num_leaves | 31 | 15, 63 |

### Ablation Study Suggestions

```python
# Remove latent features, keep {reconstruction_error}
X_minimal = X_hybrid[list(otu_clr.columns) + ['reconstruction_error']]

# Remove reconstruction_error, keep latent features
X_latents_only = X_hybrid[list(otu_clr.columns) + [f'latent_{i:02d}' for i in range(64)]]

# Baseline: raw OTUs only
X_baseline = otu_clr.copy()

# Compare CV performance across all variants
```

### Integration with Existing Notebooks

- **`hybrid_stacking_cvd.ipynb`**: Linear regression age → GAI approach
- **`gai_classifier_compact.ipynb`**: Minimal, focused 1D GAI pipeline
- **`thick_residual_hybrid_stacking.ipynb`** (this): Neural + tree ensemble approach

Choose based on:
- **Speed**: Use `gai_classifier_compact.ipynb`
- **Reproducibility**: Use `hybrid_stacking_cvd.ipynb` with SHAP
- **Maximum performance**: Use `thick_residual_hybrid_stacking.ipynb`